# Variable de control
**Call asiática aritmética** usando como variable de control la **call asiática geométrica**.

Datos:
- S0 = 100
- r = 5%
- sigma = 20%
- K = 100
- T = 1
- 100 rutas



In [2]:
import numpy as np
from scipy.stats import norm

## Datos 

In [3]:
S0 = 100
r = 0.05
sigma = 0.20
K = 100
T = 1
n_rutas = 100
n_pasos = 252        # pasos diarios
dt = T / n_pasos

np.random.seed(42)

## 100 rutas
En cada ruta se simula el precio con movimiento browniano geométrico:

$S_{t+dt} = S_t \cdot e^{(r - \sigma^2/2)dt + \sigma \sqrt{dt} Z}$

y se calculan los dos payoffs descontados.

In [12]:
Y = np.zeros(n_rutas)   # asiatica aritmetica
X = np.zeros(n_rutas)   # asiatica geometrica

for i in range(n_rutas):
    S = S0
    precios = []
    for j in range(n_pasos):
        z = np.random.normal()
        S = S * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z)
        precios.append(S)
    precios = np.array(precios)

    prom_arit = np.mean(precios)
    prom_geom = np.exp(np.mean(np.log(precios)))

    Y[i] = np.exp(-r * T) * max(prom_arit - K, 0)
    X[i] = np.exp(-r * T) * max(prom_geom - K, 0)

print("ruta      y         x")
for i in range(10):
    print(i + 1, "   ", round(Y[i], 4), "   ", round(X[i], 4))
print("...")

ruta      y         x
1     1.0431     0.9091
2     26.3409     24.7002
3     0.0     0.0
4     20.8798     19.8547
5     15.4587     14.8623
6     12.2373     12.0551
7     1.6347     1.4466
8     0.0     0.0
9     0.0     0.0
10     0.9058     0.7367
...


## E[X] conocida (fórmula cerrada de la asiática geométrica)
Para la asiática geométrica discreta con n fechas, el log del promedio geométrico es normal con:

$\mu_G = \ln S_0 + (r - \sigma^2/2) \, T \frac{n+1}{2n}$

$\sigma_G = \sigma \sqrt{T \frac{(n+1)(2n+1)}{6n^2}}$

$E[X] = e^{-rT}\left[ e^{\mu_G + \sigma_G^2/2} N(d_1) - K N(d_2) \right]$, con $d_2 = \frac{\mu_G - \ln K}{\sigma_G}$, $d_1 = d_2 + \sigma_G$

In [5]:
n = n_pasos
mu_G = np.log(S0) + (r - 0.5 * sigma**2) * T * (n + 1) / (2 * n)
sigma_G = sigma * np.sqrt(T * (n + 1) * (2 * n + 1) / (6 * n**2))

d2 = (mu_G - np.log(K)) / sigma_G
d1 = d2 + sigma_G

EX = np.exp(-r * T) * (np.exp(mu_G + 0.5 * sigma_G**2) * norm.cdf(d1) - K * norm.cdf(d2))
print("E[X] =", round(EX, 4))

E[X] = 5.5655


## Medias y tabla de desviaciones

In [6]:
y_barra = np.mean(Y)
x_barra = np.mean(X)
print("y barra =", round(y_barra, 4))
print("x barra =", round(x_barra, 4))

dx = X - x_barra          # xi - x barra
dy = Y - y_barra          # yi - y barra
dxdy = dx * dy            # (xi - x barra)(yi - y barra)
dx2 = dx**2               # (xi - x barra)^2

print()
print("ruta   xi-xbar    yi-ybar    producto    (xi-xbar)^2")
for i in range(10):
    print(i + 1, "  ", round(dx[i], 4), "  ", round(dy[i], 4), "  ", round(dxdy[i], 4), "  ", round(dx2[i], 4))
print("...")
print("Suma producto =", round(np.sum(dxdy), 4))
print("Suma (xi-xbar)^2 =", round(np.sum(dx2), 4))

y barra = 5.5604
x barra = 5.3669

ruta   xi-xbar    yi-ybar    producto    (xi-xbar)^2
1    -5.3669    -5.5604    29.8424    28.8037
2    2.2231    2.2429    4.9862    4.942
3    -5.3669    -5.5604    29.8424    28.8037
4    26.6974    28.309    755.7778    712.7534
5    19.4029    19.8361    384.879    376.4735
6    3.2845    3.8081    12.5077    10.7881
7    -2.1897    -1.8269    4.0003    4.7948
8    -5.3669    -5.5604    29.8424    28.8037
9    -5.3669    -5.5604    29.8424    28.8037
10    1.2994    1.2924    1.6794    1.6885
...
Suma producto = 5170.3436
Suma (xi-xbar)^2 = 4999.2633


## Coeficiente b
$\hat b = \frac{\sum (x_i - \bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2}$

In [7]:
b = np.sum(dxdy) / np.sum(dx2)
print("b =", round(b, 4))

b = 1.0342


## Varianzas, covarianza y correlación

In [8]:
var_x = np.sum(dx2) / (n_rutas - 1)
sigma_x = np.sqrt(var_x)

var_y = np.sum(dy**2) / (n_rutas - 1)
sigma_y = np.sqrt(var_y)

cov_xy = np.sum(dxdy) / (n_rutas - 1)
rho = cov_xy / (sigma_x * sigma_y)

b_estrella = cov_xy / var_x

print("Var(x) =", round(var_x, 4), "   sigma x =", round(sigma_x, 4))
print("Var(y) =", round(var_y, 4), "   sigma y =", round(sigma_y, 4))
print("Cov(x,y) =", round(cov_xy, 4))
print("rho =", round(rho, 4))
print("b* = Cov/Var(x) =", round(b_estrella, 4))

Var(x) = 50.4976    sigma x = 7.1062
Var(y) = 54.043    sigma y = 7.3514
Cov(x,y) = 52.2257
rho = 0.9997
b* = Cov/Var(x) = 1.0342


## reduce la varianza?

Se reduce si $b^2 \sigma_x^2 < 2 b \sigma_x \sigma_y \rho_{xy}$

In [9]:
izq = b**2 * var_x
der = 2 * b * sigma_x * sigma_y * rho
print("b^2 * sigma_x^2 =", round(izq, 4))
print("2 b sigma_x sigma_y rho =", round(der, 4))

if izq < der:
    print("Si reduce la varianza")
else:
    print("No reduce la varianza")

b^2 * sigma_x^2 = 54.0129
2 b sigma_x sigma_y rho = 108.0258
Si reduce la varianza


## variaanza con variable de control
$Var[Y(b)] = \sigma_y^2 + b^2 \sigma_x^2 - 2 b \sigma_x \sigma_y \rho$

In [10]:
var_control = var_y + b**2 * var_x - 2 * b * sigma_x * sigma_y * rho

print("Var normal:", round(var_y, 4))
print("Var con control:", round(var_control, 4))
print("Reduccion de varianza:", round((1 - var_control / var_y) * 100, 2), "%")

Var normal: 54.043
Var con control: 0.0301
Reduccion de varianza: 99.94 %


## Precio estimado con variable de control
$\hat y = \bar y - \hat b(\bar x - E[X])$

In [11]:
print("E[X] =", round(EX, 4))
print("x barra - E[X] =", round(x_barra - EX, 4))
print("b(x barra - E[X]) =", round(b * (x_barra - EX), 4))

y_control = y_barra - b * (x_barra - EX)

print()
print("Precio Monte Carlo normal:", round(y_barra, 4))
print("Precio con variable de control:", round(y_control, 4))

E[X] = 5.5655
x barra - E[X] = -0.1986
b(x barra - E[X]) = -0.2054

Precio Monte Carlo normal: 5.5604
Precio con variable de control: 5.7658


## Conclusión
La asiática geométrica esta casi perfectamente correlacionada con la aritmetica (rho ≈ 1), por eso funciona muy bien como variable de control,  con solo 100 rutas la varianza baja demasiado y el precio estimado de la call asiática aritmtica queda mucho más preciso que con Monte Carlo normal.